# Write_File Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Hello, disk.** Two `.write()` calls, each line ended by an explicit `\n`, verified by reading the file straight back.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

with open("sample_data/note.txt", "w", encoding="utf-8") as f:
    f.write("Dear notebook,\n")
    f.write("Writing files is easier than it sounds.\n")

print(Path("sample_data/note.txt").read_text(encoding="utf-8"))

**2. Counting characters.** `.write()` returns the character count, and the three counts sum to exactly `len()` of the file.

In [ ]:
from pathlib import Path

with open("sample_data/counted_note.txt", "w", encoding="utf-8") as f:
    first  = f.write("Files ")
    second = f.write("remember ")
    third  = f.write("\n")

total = first + second + third
print("characters written:", total)          # 16
print("len() agrees:", total ==
      len(Path("sample_data/counted_note.txt").read_text(encoding="utf-8")))

**3. Numbers are not welcome.** Text mode refuses ints with `TypeError`; convert with `str()` or let an f-string convert (and format) for you.

In [ ]:
from pathlib import Path

score = 95
rate = 0.256

try:
    with open("sample_data/rejected_score.txt", "w", encoding="utf-8") as f:
        f.write(score)                       # int, not str!
except TypeError as e:
    print("Rejected ->", e)

with open("sample_data/score.txt", "w", encoding="utf-8") as f:
    f.write("Score: " + str(score) + "\n")   # fix 1: str()
    f.write(f"Bonus rate: {rate:.0%}\n")     # fix 2: f-string formats too

print(Path("sample_data/score.txt").read_text(encoding="utf-8"))
# Score: 95
# Bonus rate: 26%

## Part 2 — Practice

**4. The silent overwrite trap.** The second `open(..., "w")` emptied the file the moment it succeeded — before a single character was written.

In [ ]:
from pathlib import Path

with open("sample_data/version.txt", "w", encoding="utf-8") as f:
    f.write("Version 1: precious results\n")
print(Path("sample_data/version.txt").read_text(encoding="utf-8"))

with open("sample_data/version.txt", "w", encoding="utf-8") as f:
    f.write("Version 2\n")
print(Path("sample_data/version.txt").read_text(encoding="utf-8"))

# Version 1 died the instant the SECOND open(..., "w") succeeded:
# "w" empties the file before anything new is written.

**5. Glued shopping list.** Without `\\n` the items concatenate into one token; `item + "\\n"` gives one proper line per item.

In [ ]:
from pathlib import Path

items = ["rice", "eggs", "milk"]

with open("sample_data/glued.txt", "w", encoding="utf-8") as f:
    for item in items:
        f.write(item)                        # forgot the \n!

print(repr(Path("sample_data/glued.txt").read_text(encoding="utf-8")))
# 'riceeggsmilk'

with open("sample_data/tidy.txt", "w", encoding="utf-8") as f:
    for item in items:
        f.write(item + "\n")

print(Path("sample_data/tidy.txt").read_text(encoding="utf-8"))

**6. `writelines()` does not mean lines.** It is really "write each": it inserts nothing between items, so the `\\n` must already be inside your strings.

In [ ]:
from pathlib import Path

players = ["Sarah", "Tanvir", "Amina"]

with open("sample_data/glued_players.txt", "w", encoding="utf-8") as f:
    f.writelines(players)                    # adds NOTHING between items

print(repr(Path("sample_data/glued_players.txt").read_text(encoding="utf-8")))
# 'SarahTanvirAmina'

with open("sample_data/tidy_players.txt", "w", encoding="utf-8") as f:
    f.writelines(p + "\n" for p in players)

print(Path("sample_data/tidy_players.txt").read_text(encoding="utf-8"))

**7. Born blank.** `"w"` creates a missing file for free — and wipes an existing one just as freely.

In [ ]:
from pathlib import Path

fresh = Path("sample_data", "auto_created.txt")
print("Before:", fresh.exists())             # False

with open(fresh, "w", encoding="utf-8") as f:
    f.write("Born just now, thanks to 'w'.\n")

print("After :", fresh.exists())             # True
print(fresh.read_text(encoding="utf-8"))

# Missing file -> created. Existing file -> silently wiped first.

## Part 3 — Challenge

**8. Report card builder.** Compose the document as a list of lines, `"\\n".join()` them, write once, and verify by reading back.

In [ ]:
from pathlib import Path

scores = {"Sarah": 92, "Tanvir": 78, "Amina": 95}
average = sum(scores.values()) / len(scores)

lines = []
lines.append("PYTHON QUIZ - REPORT CARD")
lines.append("=" * 25)
for name, score in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
    lines.append(f"{name:<8} {score:>3} {'#' * (score // 10)}")
lines.append("-" * 25)
lines.append(f"Class average : {average:.1f}")

with open("sample_data/report_card.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print(Path("sample_data/report_card.txt").read_text(encoding="utf-8"))

**9. Save–load round-trip.** Save one number per line, then rebuild the integers by stripping each line and calling `int()`.

In [ ]:
marks = [88, 92, 79]

# SAVE: one number per line, so reading back is trivial.
with open("sample_data/marks.txt", "w", encoding="utf-8") as out:
    for m in marks:
        out.write(f"{m}\n")

# LOAD: reconstruct the numbers from disk.
loaded = []
with open("sample_data/marks.txt", encoding="utf-8") as inp:
    for line in inp:
        loaded.append(int(line.strip()))

print("saved :", marks)
print("loaded:", loaded, "-> total", sum(loaded))